# DV-05 TabM/RealMLP on Colab GPU
**Goal:** Train a genuinely diverse model family (TabM/RealMLP) for pool injection (DV-04) and hill climbing (DV-11).

**Expected OOF:** 0.967-0.969 (vs 0.940 for old MLP)
**Decorrelation:** corr ~0.906-0.986 vs GBDT pack (pubmk_nn was 0.906, lookup was 0.986)
**Compute:** Colab T4 GPU (~30-45 min)

---

## Setup Instructions
1. **Runtime → Change runtime type → T4 GPU**
2. **Run all cells** (Ctrl+F9)
3. **First run:** Authenticate with Kaggle when prompted
4. **Live progress:** Watch epoch bars below (tqdm)
5. **Output:** CSVs auto-download at end

In [ ]:
# ┃┃┃ 1. Install & Authenticate ┃┃┃
!pip install -q kaggle pytabkit tqdm 2>/dev/null

# Kaggle auth - run once, then comment out
from google.colab import userdata
import os, json

# Option A: Use Colab Secrets (recommended)
# Add KAGGLE_USERNAME and KAGGLE_KEY in Colab Secrets (🔑 icon)
if 'KAGGLE_USERNAME' in userdata.get_all():
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print("✅ Using Colab Secrets for Kaggle auth")
else:
    # Option B: Upload kaggle.json manually
    from google.colab import files
    print("📤 Upload your kaggle.json file...")
    uploaded = files.upload()
    for fn, content in uploaded.items():
        with open('/tmp/kaggle.json', 'wb') as f:
            f.write(content)
    os.makedirs('/root/.kaggle', exist_ok=True)
    os.system('cp /tmp/kaggle.json /root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print("✅ Kaggle credentials configured")

# Verify auth
!kaggle config view

In [ ]:
# ┃┃┃ 2. Download Data from Kaggle ┃┃┃
import os
from pathlib import Path

DATA_DIR = Path('/content/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Download competition files
print("📥 Downloading competition data...")
!kaggle competitions download -c playground-series-s6e8 -p {DATA_DIR} --force 2>&1 | tail -5

# Unzip
import zipfile
for zip_file in DATA_DIR.glob('*.zip'):
    print(f"📦 Extracting {zip_file.name}...")
    with zipfile.ZipFile(zip_file, 'r') as z:
        z.extractall(DATA_DIR)
    zip_file.unlink()  # remove zip after extract

# Verify
for f in ['train.csv', 'test.csv', 'sample_submission.csv']:
    fp = DATA_DIR / f
    print(f"  {'✅' if fp.exists() else '❌'} {f}: {fp.stat().st_size/1e6:.1f} MB" if fp.exists() else f"  ❌ {f}: MISSING")

In [ ]:
# ┃┃┃ 3. Setup & Imports ┃┃┃
import sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"?횽️  Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"?풾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.cuda.empty_cache()

DATA_DIR = Path('/content/data')
OUT_DIR = Path('/content/output')
OUT_DIR.mkdir(parents=True, exist_ok=True)

CAT_COLS = ['gender', 'stress_level', 'academic_work_impact']
NUM_COLS_RAW = [
    'age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
    'work_study_hours', 'sleep_hours', 'notifications_per_day',
    'app_opens_per_day', 'weekend_screen_time'
]

SEED = 42
N_FOLDS = 5
N_SEEDS = 5
MAX_EPOCHS = 150
BATCH_SIZE = 1024
PATIENCE = 15

def prog(msg):
    print(f"[DV05] {msg}", flush=True)

In [ ]:
# ┃┃┃ 4. Feature Engineering (Winning Feature Set) ┃┃┃
def add_missingness_indicators(df, num_cols):
    df = df.copy()
    for col in num_cols:
        df[f'{col}_was_missing'] = df[col].isna().astype(float)
    return df

def add_pool141_features(df):
    df = df.copy()
    df['total_screen_time'] = df['daily_screen_time_hours'] + df['weekend_screen_time']
    df['screen_per_notification'] = df['daily_screen_time_hours'] / (df['notifications_per_day'] + 1e-6)
    df['screen_per_app_open'] = df['daily_screen_time_hours'] / (df['app_opens_per_day'] + 1e-6)
    df['notif_per_app_open'] = df['notifications_per_day'] / (df['app_opens_per_day'] + 1e-6)
    df['leisure_hours'] = df['social_media_hours'] + df['gaming_hours']
    df['leisure_productive_ratio'] = df['leisure_hours'] / (df['work_study_hours'] + 1e-6)
    df['sleep_debt'] = np.maximum(8 - df['sleep_hours'], 0)
    df['screen_per_sleep_hour'] = df['daily_screen_time_hours'] / (df['sleep_hours'] + 1e-6)
    df['age_screen_interaction'] = df['age'] * df['daily_screen_time_hours']
    df['age_group'] = pd.cut(df['age'], bins=[0, 18, 25, 35, 50, 100], labels=[0, 1, 2, 3, 4]).astype(float)
    df['weekend_weekday_ratio'] = df['weekend_screen_time'] / (df['daily_screen_time_hours'] + 1e-6)
    df['weekend_excess'] = df['weekend_screen_time'] - df['daily_screen_time_hours']
    stress_map = {'Low': 0, 'Medium': 1, 'High': 2, 'Unknown': 1}
    impact_map = {'No': 0, 'Yes': 1, 'Unknown': 0}
    for col, mapping in [('stress_level', stress_map), ('academic_work_impact', impact_map)]:
        if df[col].dtype == 'object':
            df[f'{col}_num'] = df[col].map(mapping)
        else:
            df[f'{col}_num'] = df[col]
    df['stress_screen'] = df['stress_level_num'] * df['daily_screen_time_hours']
    df['impact_screen'] = df['academic_work_impact_num'] * df['daily_screen_time_hours']
    return df

def lattice_target_encoding(train, test, cat_cols, smoothing=20, min_samples_leaf=10):
    train = train.copy()
    test = test.copy()
    global_mean = train['addicted_label'].mean()
    
    for col in cat_cols:
        if col not in train.columns:
            continue
        le = LabelEncoder()
        train[col] = train[col].fillna('Unknown').astype(str)
        test[col] = test[col].fillna('Unknown').astype(str)
        train[f'{col}_le'] = le.fit_transform(train[col])
        test[f'{col}_le'] = le.transform(test[col])
        
        stats = train.groupby(f'{col}_le')['addicted_label'].agg(['mean', 'count'])
        stats['smooth'] = (stats['mean'] * stats['count'] + global_mean * smoothing) / (stats['count'] + smoothing)
        stats.loc[stats['count'] < min_samples_leaf, 'smooth'] = global_mean
        
        train[f'{col}_te'] = train[f'{col}_le'].map(stats['smooth']).fillna(global_mean)
        test[f'{col}_te'] = test[f'{col}_le'].map(stats['smooth']).fillna(global_mean)
    
    # Pair lattice TE (all pairs of binned numerics)
    num_cols = [c for c in train.columns if c not in ['id', 'addicted_label'] + cat_cols 
                and train[c].dtype in ['float64', 'int64', 'float32', 'int32']]
    
    for col in num_cols:
        if train[col].nunique() > 20:
            bins = pd.qcut(train[col], q=10, labels=False, duplicates='drop')
            train[f'{col}_bin'] = bins.astype(float)
            test[f'{col}_bin'] = pd.cut(test[col], bins=10, labels=False).astype(float)
        else:
            train[f'{col}_bin'] = train[col]
            test[f'{col}_bin'] = test[col]
    
    bin_cols = [c for c in train.columns if c.endswith('_bin')]
    
    prog(f"🔗 Creating {len(bin_cols)*(len(bin_cols)-1)//2} pair lattice features...")
    for i, c1 in enumerate(tqdm(bin_cols, desc="Lattice TE")):
        for c2 in bin_cols[i+1:]:
            pair_name = f'pair_{c1}_{c2}'
            train[pair_name] = train[c1].astype(str) + '_' + train[c2].astype(str)
            test[pair_name] = test[c1].astype(str) + '_' + test[c2].astype(str)
            
            stats = train.groupby(pair_name)['addicted_label'].agg(['mean', 'count'])
            stats['smooth'] = (stats['mean'] * stats['count'] + global_mean * smoothing) / (stats['count'] + smoothing)
            stats.loc[stats['count'] < min_samples_leaf, 'smooth'] = global_mean
            
            train[f'{pair_name}_te'] = train[pair_name].map(stats['smooth']).fillna(global_mean)
            test[f'{pair_name}_te'] = test[pair_name].map(stats['smooth']).fillna(global_mean)
            
            train.drop(columns=[pair_name], inplace=True)
            test.drop(columns=[pair_name], inplace=True)
    
    return train, test

def preprocess_basic(df):
    for col in CAT_COLS:
        df[col] = df[col].fillna('Unknown')
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
    return df

def get_features(train, test):
    num_cols = [c for c in NUM_COLS_RAW if c in train.columns]
    train = add_missingness_indicators(train, num_cols)
    test = add_missingness_indicators(test, num_cols)
    train = add_pool141_features(train)
    test = add_pool141_features(test)
    train, test = lattice_target_encoding(train, test, CAT_COLS)
    train = preprocess_basic(train)
    test = preprocess_basic(test)
    feature_cols = [c for c in train.columns if c not in ['id', 'addicted_label']]
    X = train[feature_cols]
    y = train['addicted_label'].values.astype(np.float32)
    X_test = test[feature_cols]
    return X, y, X_test, feature_cols

# Load & featurize
prog("📊 Loading data...")
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
prog(f"Train: {train.shape}, Test: {test.shape}")

prog("🔧 Feature engineering...")
X, y, X_test, feature_cols = get_features(train, test)
prog(f"✅ Features: {len(feature_cols)}, Train: {len(X)}, Test: {len(X_test)}")

In [ ]:
# ┃┃┃ 5. TabM / RealMLP Training with LIVE Progress ┃┃┃
import pytabkit as ptk
from pytabkit.models import TabM, RealMLP
from pytabkit.models.nn_models.tabm import TabMConfig
from pytabkit.models.nn_models.realmlp import RealMLPConfig

prog(f"📦 pytabkit version: {ptk.__version__}")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(skf.split(np.zeros(len(y)), y))

class ProgressCallback:
    def __init__(self, model_name, seed_idx, fold_idx, total_seeds, total_folds):
        self.model_name = model_name
        self.seed_idx = seed_idx
        self.fold_idx = fold_idx
        self.total_seeds = total_seeds
        self.total_folds = total_folds
        self.epoch = 0
        self.best_auc = 0
        self.pbar = None
    
    def on_epoch_end(self, epoch, logs=None):
        self.epoch = epoch
        if logs and 'val_auc' in logs:
            val_auc = logs['val_auc']
            if val_auc > self.best_auc:
                self.best_auc = val_auc
            if self.pbar:
                self.pbar.set_postfix({'val_auc': f'{val_auc:.5f}', 'best': f'{self.best_auc:.5f}'})
                self.pbar.update(1)

def train_model(X_train, y_train, X_val, y_val, X_test, seed=42, model_type='tabm', callback=None):
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)
    
    X_train_t = torch.tensor(X_train_s, dtype=torch.float32).to(DEVICE)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).to(DEVICE)
    X_val_t = torch.tensor(X_val_s, dtype=torch.float32).to(DEVICE)
    y_val_t = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)
    X_test_t = torch.tensor(X_test_s, dtype=torch.float32).to(DEVICE)
    
    if model_type == 'tabm':
        config = TabMConfig(
            n_blocks=10,
            d_block=160,
            dropout=0.02,
            lr=5e-4,
            n_epochs=MAX_EPOCHS,
            batch_size=BATCH_SIZE,
            weight_decay=1e-4,
            device=DEVICE,
            seed=seed,
        )
        model = TabM(config)
    else:
        config = RealMLPConfig(
            n_blocks=8,
            d_block=256,
            dropout=0.1,
            lr=1e-3,
            n_epochs=MAX_EPOCHS,
            batch_size=BATCH_SIZE,
            weight_decay=1e-4,
            device=DEVICE,
            seed=seed,
        )
        model = RealMLP(config)
    
    # Train with progress
    if callback:
        callback.pbar = tqdm(total=MAX_EPOCHS, 
                           desc=f"{model_type.upper()} S{callback.seed_idx+1}/{callback.total_seeds} F{callback.fold_idx+1}/{callback.total_folds}",
                           leave=False)
    
    model.fit(X_train_t, y_train_t, X_val=X_val_t, y_val=y_val_t, callbacks=[callback] if callback else None)
    
    if callback and callback.pbar:
        callback.pbar.close()
    
    val_pred = model.predict_proba(X_val_t)[:, 1].cpu().numpy()
    test_pred = model.predict_proba(X_test_t)[:, 1].cpu().numpy()
    return val_pred, test_pred

In [ ]:
# ┃┃┃ 6. Main Training Loop with Live Progress ┃┃┃
model_types = ['tabm', 'realmlp']
best_model_type = None
best_cv = 0
best_oof = None
best_test = None

t_start = time.time()

for model_type in model_types:
    prog(f"\n{'='*60}")
    prog(f"🚀 Training {model_type.upper()}")
    prog(f"{'='*60}")
    
    oof_seeds, test_seeds = [], []
    
    for s_i, seed in enumerate(range(SEED, SEED + N_SEEDS)):
        oof = np.zeros(len(y))
        test_pred = np.zeros(len(X_test))
        
        fold_pbar = tqdm(total=N_FOLDS, desc=f"{model_type.upper()} Seed {s_i+1}/{N_SEEDS}", position=0, leave=True)
        
        for f_i, (tr_idx, va_idx) in enumerate(splits):
            callback = ProgressCallback(model_type, s_i, f_i, N_SEEDS, N_FOLDS)
            
            val_pred, test_fold_pred = train_model(
                X.iloc[tr_idx].values, y[tr_idx],
                X.iloc[va_idx].values, y[va_idx],
                X_test.values,
                seed=seed,
                model_type=model_type,
                callback=callback
            )
            
            oof[va_idx] = val_pred
            test_pred += test_fold_pred / N_FOLDS
            fold_pbar.update(1)
            fold_pbar.set_postfix({'fold_auc': f'{roc_auc_score(y[va_idx], val_pred):.5f}'})
        
        fold_pbar.close()
        
        cv_auc = roc_auc_score(y, oof)
        prog(f"  ✅ Seed {s_i+1}/{N_SEEDS} DONE | OOF AUC: {cv_auc:.5f} | Elapsed: {(time.time()-t_start)/60:.1f} min")
        oof_seeds.append(oof)
        test_seeds.append(test_pred)
    
    oof_final = np.mean(oof_seeds, axis=0)
    test_final = np.clip(np.mean(test_seeds, axis=0), 0, 1)
    cv_auc = roc_auc_score(y, oof_final)
    
    prog(f"  🏁 {model_type.upper()} FINAL: OOF AUC = {cv_auc:.5f} | Test Mean = {test_final.mean():.5f}")
    
    if cv_auc > best_cv:
        best_cv = cv_auc
        best_model_type = model_type
        best_oof = oof_final
        best_test = test_final

In [ ]:
# ┃┃┃ 7. Save & Auto-Download ┃┃┃
prog(f"\n{'='*60}")
prog(f"🏆 BEST: {best_model_type.upper()} with CV AUC = {best_cv:.5f}")
prog(f"{'='*60}")

# Self-checks
assert not np.isnan(best_oof).any() and not np.isnan(best_test).any(), "NaN in predictions"
assert best_oof.min() >= 0 and best_oof.max() <= 1, "OOF out of [0,1]"
assert len(best_test) == len(test) == 296302, "wrong test length"
mean_gap = abs(best_test.mean() - pd.Series(y).mean())
prog(f"📊 Test Mean: {best_test.mean():.5f} | Target Mean: {pd.Series(y).mean():.5f} | Gap: {mean_gap:.4f}")

sub_id = pd.read_csv(DATA_DIR / 'sample_submission.csv')['id']

sub_path = OUT_DIR / f'submission_dv05_{best_model_type}.csv'
pd.DataFrame({'id': sub_id, 'addicted_label': best_test}).to_csv(sub_path, index=False)
prog(f"📈 Submission saved: {sub_path}")

oof_path = OUT_DIR / f'dv05_{best_model_type}_oof.csv'
pd.DataFrame({'id': train['id'], f'dv05_{best_model_type}_oof': best_oof}).to_csv(oof_path, index=False)
prog(f"📈 OOF saved: {oof_path}")

# Auto-download in Colab
from google.colab import files
prog("⬇️  Downloading files...")
files.download(str(sub_path))
files.download(str(oof_path))

prog(f"\n⏱️  Total time: {(time.time()-t_start)/60:.1f} min")
prog("✅ Done! Files downloaded to your machine.")
prog("📤 Next: Upload CSVs to Kaggle and submit via CLI")